In [ ]:
# Purpose - plots cutaways for the big boy (141)

findex = 250 #-7
%load_ext autoreload
%autoreload


In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../../v2_proc_files')
from useful_funcs import *
from useful_funcs import fq_cut
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper

plt.close('all')
plt.ioff() 

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

steps = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps}[steps_choice]
steps = list(range(270, 416, 1))

boilerplate = "NOT FILLED IN"

print(steps)

In [ ]:
# Configure Data

d = run.config_data(steps, "fields", includes_pressure = False)

In [ ]:
# write ez reps

def decimal_index(A, coord):
    ylen, xlen = A.shape
    return A[int(coord[1] * ylen), int(coord[0] * xlen)]

ez_reps = np.zeros_like(steps, dtype=float)
ez_reps_main = np.zeros_like(steps, dtype=float)
ez_reps_left = np.zeros_like(steps, dtype=float)
ez_reps_right = np.zeros_like(steps, dtype=float)

for i in range(len(steps)):
    print(f"Processing step {i}")
    buffer = 10
    ybuffer = 450

    left_x_center = run.xlim / 2
    right_x_center = run.xlim / 2
    right_y_center = 1300

    square_size = 40

    #main_rect = [0 + buffer, buffer + square_size, run.ylim / 2 - square_size / 2, run.ylim / 2 + square_size / 2]
    main_rect = [20, 100, 3000, 4000]
    left_rect = [left_x_center - 200, left_x_center + 200, 1200 - 300, 2000 - 300]
    right_rect = [right_x_center - 120, right_x_center + 120, right_y_center - 240, right_y_center + 240]
    #roi = fq_cut(d.fields, i, [0 + buffer, run.xlim - buffer, 0 + ybuffer, run.ylim - ybuffer])
    #ez_reps[i] = roi.ez.mean()

    square_size = 40

    main_roi = fq_cut(d.fields, i, main_rect)
    left_roi = fq_cut(d.fields, i, left_rect)
    right_roi = fq_cut(d.fields, i, right_rect)

    ez_reps_main[i] = main_roi.ez.mean() #0.0088 #main_roi.ez.mean()
    ez_reps_left[i] = right_roi.ez.mean() #left_roi.ez.mean() #0.0088 #
    ez_reps_right[i] = right_roi.ez.mean()


np.save(f"{run.path}/ez_reps_main.npy", ez_reps_main)
np.save(f"{run.path}/ez_reps_left.npy", ez_reps_left)
np.save(f"{run.path}/ez_reps_right.npy", ez_reps_right)
#
#ez_rep = roi.ez.mean() * skipfq_cut(d.fields, rel_ts, [0 + buffer, run.xlim - buffer, 0 + ybuffer, run.ylim - ybuffer])

In [ ]:
ez_reps_main = np.load(f"{run.path}/ez_reps_main.npy")
ez_reps_left = np.load(f"{run.path}/ez_reps_left.npy")
ez_reps_right = np.load(f"{run.path}/ez_reps_right.npy")

plt.close('all')

plt.plot(steps, ez_reps_main, label = "Main")
plt.plot(steps, ez_reps_left, label = "Left")
plt.plot(steps, ez_reps_right, label = "Right")
plt.legend()
plt.show()

In [ ]:
print(0 / 0)

In [ ]:
# Choose Regions of Interest

xlim = run.xlim
ylim = run.ylim

#rect = [xlim / 2 - 4, xlim / 2 + 4, ylim / 2 - 10, ylim / 2 + 10]
#rect = [0, xlim, 0, ylim]
#rect = [50, 70, 130, 190]
rect = [0, xlim, 0, ylim]
rect = [5, xlim - 5, ylim / 2 - 240, ylim / 2 + 240]

tchosen = 0
roi = fq_cut(d.fields, tchosen, rect)

#xcut_y = 164.4
xcut_y = ylim / 10 #5950
ycut_x = xlim / 2

xstrip = fq_xcut(roi, xcut_y)
ystrip = fq_ycut(roi, ycut_x)

In [ ]:
roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim])

#roi = fq_cut(d.fields, 0, [55, 65, 110, 130])

# Create a subplot with one plot
plt.figure()
fig, ax = plt.subplots()

#xa, ya = [0.02, 0.99]

#ax.text(xa, ya, f"$\sigma$ = {run.sigma}", color='black', ha='left', va='top',
#        transform=ax.transAxes, fontsize = 2, weight = "bold",
#        bbox=dict(boxstyle='square,pad=0.3', edgecolor='white', facecolor='white', alpha=0.8))

#plt.title(f"{run.boilerplate}")


mesh = plot_fq3_flipped(roi, roi.ex, plot = ax, vabsmax = 0.1, positive = False, downsample = 5, cmap = 'seismic')

buffer = 5
yextent = 300 
#plt.xlim(buffer, run.xlim - buffer)
#plt.ylim(0, run.ylim)
#rect_patch = plt.Rectangle((rect[0], rect[2]), rect[1] - rect[0], rect[3] - rect[2], 
#                           linewidth=2, edgecolor='red', facecolor='none', linestyle=':')
#ax.add_patch(rect_patch)
#cbar = fig.colorbar(mesh, ax=ax)
#cbar.set_label("$n / n_0$", rotation=270)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
print(run.steps)

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.patches import ConnectionPatch

In [ ]:
def plot_fq3_flipped(roi, quantity, plot=None, title=None, vabsmax=-1, cmap='viridis', downsample=1, positive=False, cbar_label=None):
    is_plt = plot is None
    if is_plt:
        plot = plt

    # Downsample the data and transpose
    if downsample > 1:
        quantity = quantity[::downsample, ::downsample]  # Transpose
        roi_x = roi.x[::downsample]  # Flip axes
        roi_y = roi.y[::downsample]
    else:
        quantity = quantity  # Transpose
        roi_x = roi.x
        roi_y = roi.y

    xmin, xmax, ymin, ymax = min(roi_x), max(roi_x), min(roi_y), max(roi_y)

    flipped_extent = [ymin, ymax, xmax, xmin]

    if positive:
        mesh = plot.imshow(quantity.T, cmap=cmap, vmax=vabsmax, vmin=0, origin='upper', extent=flipped_extent)
    elif vabsmax == -1:
        mesh = plot.imshow(quantity.T, cmap=cmap, origin='upper', extent=flipped_extent)
    else:
        mesh = plot.imshow(quantity.T, cmap=cmap, vmax=vabsmax, vmin=-vabsmax, origin='upper', extent=flipped_extent)

    if is_plt:
        plt.xlabel('$y$ [$c/\omega_{p}$]')  # Flipped
        plt.ylabel('$x$ [$c/\omega_{p}$]')
        #plt.xlim(ymin, ymax)
        #plt.ylim(xmin, xmax)
        if cbar_label is None:
            plt.colorbar()
        else:
            plt.colorbar(label=cbar_label)
    else:
        plot.set_xlabel('$y$ [$c/\omega_{p}$]')
        plot.set_ylabel('$x$ [$c/\omega_{p}$]')
        #plot.set_xlim(ymin, ymax)
        #plot.set_ylim(xmin, xmax)

    #plt.gca().set_aspect('equal')
    if title is not None:
        plt.title(f"{title}")
    return mesh

In [ ]:
# read xpos



cs_file = f"{run.path}/crossings_data.npy"

cs_shift = np.load(cs_file)
cs_shift = np.concatenate((cs_shift, np.array([cs_shift[-1]]))) #hack, should not make a big difference



steps_i = range(len(cs_shift))

start = run.ylim / 2
end = 3750 # set manually by inspection to accomodate x point drift 

xpt_shift_loc = f"{run.path}/xpt_data.npy"

np.save(xpt_shift_loc, np.linspace(start, end, len(cs_shift)))
xpt_shift = np.load(xpt_shift_loc)

plt.close('all')

plt.plot(steps_i, xpt_shift)
plt.show()


print(run.path)

In [ ]:
%autoreload

# SETTINGS

#10, 10 to 2, 2

plot_choice = "hall" # "hall"
lazy = 2
contour_cheat = 2
counter = 0
buffer = 10
ybuffer = 450
lw = 2.3

var_to_plot = {'density': 'ndens', 'hall': 'ex'}[plot_choice]
cmap = {"density": "viridis", "hall": "seismic"}[plot_choice]
positive = {"density": True, "hall": False}[plot_choice]
vabsmax = {"density": 5.0, "hall": 0.1}[plot_choice]
ccolor = {"density": "white", "hall": "black"}[plot_choice]

flh1 = None
flh2 = None
flh3 = None

fig = plt.figure(figsize=(36, 12))
gs = GridSpec(2, 2, width_ratios=[1.5, 3], height_ratios=[1, 1], wspace=0.3, hspace=0.3)

for rel_ts in range(1):
    title = f"$\Omega_i t = $ {round(rel_ts * run.gyrofreq / run.mime):05d}"
    ez_rep = np.nan

    def get_crec(width, height, center = [run.xlim / 2, run.ylim / 2]):
        x1 = center[0] - width / 2
        x2 = center[0] + width / 2
        y1 = center[1] - height / 2
        y2 = center[1] + height / 2
        return [x1, x2, y1, y2]


    #rects = [[run.xlim / 2 - 200, run.xlim / 2 + 200, 1300, 1900], get_crec(240, 480)]

    left_x_center = cs_shift[steps[rel_ts]]
    right_x_center = cs_shift[steps[rel_ts]]
    right_y_center = 3300
    

    rects = [[left_x_center - 200, left_x_center + 200, 1200 - 300, 2000 - 300],
              [right_x_center - 120, right_x_center + 120, right_y_center - 240, right_y_center + 240]]
    colors = ['yellow', 'red']
    titles = ['Zoom 2', 'Zoom 1']

    def plot_main(ax):
        global flh1
        
        #roi = fq_cut(d.fields, rel_ts, [0 + buffer, run.xlim - buffer, 0, run.ylim / 2])
        roi = fq_cut(d.fields, rel_ts, [0 + buffer, run.xlim - buffer, 0 + ybuffer, run.ylim - ybuffer])
        ez_rep = roi.ez.mean()
        #ez_rep = roi.ez[0, 0]
        print(ez_rep.compute().item())

        im = plot_fq3_flipped(roi, roi[var_to_plot], plot = ax, vabsmax = vabsmax, positive = positive, downsample = lazy, cmap = cmap)

        if (flh1 is None):
            flh1 = FieldLineHelper(run, 10, contour_ds = contour_cheat)
        flh1.plot_field_lines(roi, ez_rep, plot = ax, arrow_spacing = -1, linewidth = 1, transpose = True, color = ccolor) #1000

        ax.set_ylim(run.xlim - buffer, buffer)
        ax.set_title(title)
        #ax.set_title("Main Image")
        return im

    def plot_zoom(ax, rect, color, title, flh_index):
        global flh2, flh3
        curr_flh = [flh2, flh3][flh_index]

        [y1, y2, x1, x2] = rect
        roi = fq_cut(d.fields, rel_ts, rect)
        plot_fq3_flipped(roi, roi[var_to_plot], plot = ax, vabsmax = vabsmax, positive = positive, downsample = lazy, cmap = cmap)

        if (curr_flh is None):
            if flh_index == 0:
                flh2 = FieldLineHelper(run, 8, contour_ds = contour_cheat)
                curr_flh = flh2
            else:
                flh3 = FieldLineHelper(run, 8, contour_ds = contour_cheat)  
                curr_flh = flh3
        else:
            2 + 2
            print("trying")
        curr_flh.plot_field_lines(roi, ez_rep, plot = ax, arrow_spacing = -1, transpose = True, color = ccolor)

        #ax.set_xlim(y1, y2)
        #ax.set_ylim(x1, x2)
        #ax.set_title(title)
        # Show axis ticks and labels
        ax.tick_params(axis='both', which='both', direction='in', length=4)
        ax.set_xlabel("$y$ [$c/\\omega_{p}$]")
        ax.set_ylabel("$x$ [$c/\\omega_{p}$]")
        rect_patch = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor=color, facecolor='none', lw=lw)
        return rect_patch

    def connect_rect(fig, ax_main, ax_zoom, rect, color, custom = False):
        [x1, x2, y1, y2] = rect

        if not custom:
            con1 = ConnectionPatch(xyA=(y1, x1), coordsA=ax_main.transData,
                                xyB=(0, 1), coordsB=ax_zoom.transAxes,
                                color=color, lw = lw)
        else:
            con1 = ConnectionPatch(xyA=(y1, x2), coordsA=ax_main.transData,
                                xyB=(0, 0), coordsB=ax_zoom.transAxes,
                                color=color, lw = lw)
        
        con2 = ConnectionPatch(xyA=(y2, x1), coordsA=ax_main.transData,
                                xyB=(1, 1), coordsB=ax_zoom.transAxes,
                                color=color, lw = lw)


        fig.add_artist(con1)
        fig.add_artist(con2)


    ax_main = fig.add_subplot(gs[0, :])
    im = plot_main(ax_main)
    is_custom = [False, True]

    zoom_axes = []
    for i, (rect, color, title) in enumerate(zip(rects, colors, titles)):
        ax_zoom = fig.add_subplot(gs[1, i])
        rect_patch = plot_zoom(ax_zoom, rect, color, title, i)
        ax_main.add_patch(rect_patch)
        connect_rect(fig, ax_main, ax_zoom, rect, color, custom = is_custom[i])
        zoom_axes.append(ax_zoom)

    plt.tight_layout() #maybe disable

    save_polished_fig(name = {"density": "bigrun_zooms_density_04_28_26", "hall": "bigrun_zooms_hall_04_28_26"}[plot_choice], dpi = 400)
    #save_movie_frame(counter, dpi = 200)

    
    plt.show()
    #plt.close(fig)

    print(f"Done with frame {counter}")

    counter += 1